# Garmin Health Tools Playground

This notebook lets you explore the three MCP tools exposed by the Garmin Health server.
Claude (or any MCP client) uses these tools to answer natural-language health questions by
looking up the schema and writing SQL itself — no external LLM API key required.

## Available Tools
| Tool | Purpose |
|---|---|
| **list_domains** | Return short descriptions for all 10 health data domains |
| **get_schema** | Return full table/column schema for a domain (use before writing SQL) |
| **execute_sql** | Run a `SELECT` query against the Garmin SQLite databases |


## Setup

Configure your environment and import the tools.

In [ ]:
import asyncio
import json
import pandas as pd

from garmin_mcp.server import TOOLS, handle_call_tool

# Fix for Jupyter: allow nested event loops
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "nest_asyncio"])
    import nest_asyncio
    nest_asyncio.apply()

print(f"✓ Imports successful")
print(f"✓ {len(TOOLS)} tools available: {[t.name for t in TOOLS]}")


✓ Imports successful
✓ 10 tools available


## Helper Functions

These functions make it easy to call tools and inspect results.

In [ ]:
def call_tool(tool_name: str, arguments: dict) -> dict:
    """Call an MCP tool with the given arguments dict.

    Args:
        tool_name:  One of 'list_domains', 'get_schema', 'execute_sql'.
        arguments:  Dict matching the tool's inputSchema.

    Returns:
        Parsed JSON response as a dict.
    """
    result = asyncio.run(handle_call_tool(tool_name, arguments))
    if result and len(result) > 0:
        return json.loads(result[0].text)
    return {"error": "No response from tool"}


def show_results(response: dict, limit: int = 10) -> None:
    """Display an execute_sql response in a readable format.

    Args:
        response: Tool response dict (keys: row_count, results, warning, error).
        limit:    Maximum rows to display.
    """
    print("\n" + "=" * 80)

    if response.get("error"):
        print(f"❌ ERROR: {response['error']}")
        print("=" * 80)
        return

    row_count = response.get("row_count", 0)
    results = response.get("results", [])

    print(f"✓ Query successful — {row_count} rows returned")

    if response.get("warning"):
        print(f"⚠️  {response['warning']}")

    if results:
        df = pd.DataFrame(results[:limit])
        with pd.option_context("display.max_columns", None, "display.max_colwidth", None, "display.width", None):
            print(df.to_string(index=False))
        if row_count > limit:
            print(f"\n... and {row_count - limit} more rows")
    else:
        print("(no rows returned)")

    print("=" * 80)


def list_tools() -> None:
    """Display all available tools with descriptions."""
    print("\n" + "=" * 80)
    print("AVAILABLE TOOLS")
    print("=" * 80)
    for i, tool in enumerate(TOOLS, 1):
        print(f"\n{i}. {tool.name}")
        print(f"   {tool.description[:120]}")
    print("\n" + "=" * 80)


print("✓ Helper functions defined")


✓ Helper functions defined


In [ ]:
# Quick sanity check: call list_domains to verify the server module loads correctly
domains = call_tool("list_domains", {})
print(f"Server is up — {len(domains)} domains available:")
for name, desc in domains.items():
    print(f"  {name:20s}  {desc[:60]}")


DEBUG: Configuration Check
  API Key Set: Yes
  Tools Available: 10

Raw Response:
{
  "sql": "SELECT *\nFROM sleep\nWHERE day >= strftime('%Y-%m-%d', '2026-03-29', '-7 days')\nORDER BY day DESC",
  "row_count": 6,
  "results": [
    {
      "day": "2026-03-27",
      "start": "2026-03-26 22:59:53.000000",
      "end": "2026-03-27 06:38:53.000000",
      "total_sleep": "07:39:00.000000",
      "deep_sleep": "01:43:00.000000",
      "light_sleep": "04:07:00.000000",
      "rem_sleep": "01:49:00.000000",
      "awake": "00:00:00.000000",
      "avg_spo2": null,
      "avg_rr": 17.0,
      "avg_stress": 11.0,
      "score": 94,
      "qualifier": "EXCELLENT"
    },
    {
      "day": "2026-03-26",
      "start": "2026-03-26 00:32:00.000000",
      "end": "2026-03-26 09:00:00.000000",
      "total_sleep": "08:27:00.000000",
      "deep_sleep": "01:18:00.000000",
      "light_sleep": "06:10:00.000000",
      "rem_sleep": "01:00:00.000000",
      "awake": "00:01:00.000000",
      "avg_spo2":

## Explore the Tools

Run the sections below to exercise each of the three tools.


### 1. list_domains

Returns a mapping of domain name → short description for all 10 available health domains.
No parameters needed.


In [ ]:
domains = call_tool("list_domains", {})
print(json.dumps(domains, indent=2))



✓ Query successful
📊 Results: 1 rows

Generated SQL:
--------------------------------------------------------------------------------
SELECT SUM(CAST(SUBSTR(total_sleep,1,2) AS INTEGER) + CAST(SUBSTR(total_sleep,4,2) AS INTEGER)/60.0) FROM sleep WHERE day BETWEEN strftime('%Y-%m-%d', 'now', '-7 days') AND strftime('%Y-%m-%d', 'now', '-1 day')
--------------------------------------------------------------------------------

Results:
    [SUM]
46.166667



### 2. get_schema

Returns the full table/column schema for a domain, plus the `db` and `attach_dbs` values
you need to pass to `execute_sql`. Always call this before writing SQL for a domain.


In [ ]:
# Schema for the sleep domain
sleep_schema = call_tool("get_schema", {"domain": "sleep"})
print(f"Primary DB : {sleep_schema['db']}")
print(f"Attach DBs : {sleep_schema['attach_dbs']}")
print(f"\nSchema:\n{sleep_schema['schema']}")


In [ ]:
# Schema for the activities domain (uses a different primary database)
activities_schema = call_tool("get_schema", {"domain": "activities"})
print(f"Primary DB : {activities_schema['db']}")
print(f"Attach DBs : {activities_schema['attach_dbs']}")
print(f"\nSchema:\n{activities_schema['schema']}")


### 3. execute_sql

Runs a `SELECT` statement against the Garmin SQLite databases.
Pass the `db` and `attach_dbs` values you got from `get_schema`.
Only `SELECT` is allowed — any DML/DDL is rejected.

#### Simple queries


In [ ]:
# Simple SELECT with a date filter — change the date range to match your data
result = call_tool("execute_sql", {
    "db": "garmin",
    "sql": "SELECT day, total_sleep_seconds / 3600.0 AS hours FROM sleep ORDER BY day DESC LIMIT 7",
})
show_results(result)


In [ ]:
# Aggregation: average resting heart rate by month
result = call_tool("execute_sql", {
    "db": "garmin",
    "sql": """
        SELECT
            strftime('%Y-%m', day) AS month,
            ROUND(AVG(resting_heart_rate), 1) AS avg_rhr
        FROM resting_heart_rate
        WHERE day >= date('now', '-12 months')
        GROUP BY month
        ORDER BY month
    """,
})
show_results(result)


#### JOIN across databases (attach_dbs)

Some domains span multiple SQLite files. Use `attach_dbs` to make cross-DB joins possible.


In [ ]:
# Daily summary lives in garmin_summary; attach it to query alongside garmin
result = call_tool("execute_sql", {
    "db": "garmin_summary",
    "sql": """
        SELECT
            ds.day,
            ds.total_steps,
            ds.total_calories,
            ds.intensity_time_seconds / 60 AS intensity_minutes
        FROM daily_summary ds
        ORDER BY ds.day DESC
        LIMIT 10
    """,
    "attach_dbs": {},
})
show_results(result)


In [ ]:
# Query with explicit LIMIT — useful to avoid the MAX_ROWS truncation warning
result = call_tool("execute_sql", {
    "db": "garmin_activities",
    "sql": """
        SELECT activity_type, start_time, distance_meters / 1000.0 AS km
        FROM activities
        ORDER BY start_time DESC
        LIMIT 5
    """,
})
show_results(result)


#### Error handling

`execute_sql` returns an `error` key instead of raising exceptions for rejected or invalid queries.


In [ ]:
# Unknown db key → error
result = call_tool("execute_sql", {"db": "not_a_real_db", "sql": "SELECT 1"})
print(result)


In [ ]:
# Non-SELECT statement → error (safety check)
result = call_tool("execute_sql", {
    "db": "garmin",
    "sql": "DROP TABLE sleep",
})
print(result)


## Write Your Own SQL

Use the cells below to run custom queries. First call `get_schema` for your domain to get
the right `db` value and column names, then pass SQL to `execute_sql`.


In [ ]:
# Step 1: Get the schema for the domain you care about
schema = call_tool("get_schema", {"domain": "sleep"})
print(f"db={schema['db']}  attach_dbs={schema['attach_dbs']}")
print(json.dumps(schema["schema"], indent=2))


In [ ]:
# Step 2: Write and run your SQL
result = call_tool("execute_sql", {
    "db": "garmin",            # ← from schema['db'] above
    "sql": """
        SELECT *
        FROM sleep
        ORDER BY day DESC
        LIMIT 10
    """,
    # "attach_dbs": schema["attach_dbs"],  # uncomment if the domain needs attached DBs
})
show_results(result)


## Tool Reference

Run the cell below to see all available tools and their descriptions.

In [ ]:
list_tools()

## Debugging Tips

- **Discover what's available**: call `list_domains` first to see all 10 domain names.
- **Always get the schema first**: `get_schema` gives you the exact table names, column names, and which `db` key to use with `execute_sql`.
- **Date formats**: dates are stored as `'YYYY-MM-DD'`, datetimes as `'YYYY-MM-DD HH:MM:SS'`, and time durations as `'HH:MM:SS'` strings.
- **Truncated results**: if `show_results` shows a `⚠️` warning, your query returned the `MAX_ROWS` cap (default 500). Add `LIMIT` or a `WHERE` clause.
- **Check the raw response**: inspect the full dict to see all fields.

```python
response = call_tool("execute_sql", {"db": "garmin", "sql": "SELECT * FROM sleep LIMIT 3"})
print(json.dumps(response, indent=2, default=str))
```
